# Activation Capping — Trait Logprobs Visualisation

Plots OCEAN trait scores vs. activation capping fraction for a sweep run directory.
One trait is rendered at full brightness; the other four are very faint.
Choice mass is overlaid on the same y-axis.

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
# Path to the sweep run directory (contains base/, cap_+0p25/, ... sub-dirs)
RUN_DIR = "scratch/evals/ocean/trait/c_minus_activation_capping_trait_logprobs"

# Trait to highlight at full brightness.
# Use full name ("Conscientiousness") or OCEAN letter ("C").
HIGHLIGHT_TRAIT = "C"

# Interval method for error bars.  Set to None to omit.
INTERVAL = "ci95_from_bootstrap_1000"

# X-axis label
X_LABEL = "Activation Vector Limit"

# X-axis limits (set to None for auto)
X_LIM = (-2.5, 2.5)

# Alpha for faint (non-highlighted) traits
FAINT_ALPHA = 0.12

# Apply dynamic choice-mass filter (exclude samples where mass < 1/num_choices)
DYNAMIC_MASS_FILTER = True
# ──────────────────────────────────────────────────────────────────────────────

In [3]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Ensure repo root is on the path when running from dump/
repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src_dev.evals.personality.analyze_results import (
    load_sweep_data,
    IntervalMethod,
    BIG_FIVE,
    BIG_FIVE_COLORS,
    _agg_sweep,
    _draw_col_error_bars,
    _resolve_highlight,
    _set_scale_xticks,
)

print("Imports OK")

/root/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [4]:
# ── Load data ─────────────────────────────────────────────────────────────────
run_dir = Path(RUN_DIR)
# Resolve relative to repo root if not absolute
if not run_dir.is_absolute():
    run_dir = repo_root / run_dir
assert run_dir.exists(), f"Run dir not found: {run_dir}"

sweep = load_sweep_data(run_dir)
print("Evals found:", list(sweep.evals.keys()))

df = sweep.evals.get("trait_logprobs")
if df is None:
    raise ValueError(
        "No 'trait_logprobs' eval found in this run dir.  "
        f"Available: {list(sweep.evals.keys())}"
    )

print(f"Loaded {len(df)} rows, scale range: [{df['scale'].min():.3f}, {df['scale'].max():.3f}]")

KeyboardInterrupt: 

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
interval = IntervalMethod.from_str(INTERVAL) if INTERVAL else None
lit = _resolve_highlight([HIGHLIGHT_TRAIT])

trait_agg = _agg_sweep(
    df, BIG_FIVE,
    interval=interval,
    dynamic_mass_filter=DYNAMIC_MASS_FILTER,
)
scales = trait_agg["scale"].values

has_choice_mass = "_choice_mass" in df.columns and df["_choice_mass"].notna().any()

fig, ax = plt.subplots(figsize=(12, 5.5))

# ── Trait lines ───────────────────────────────────────────────────────────────
for trait in BIG_FIVE:
    color = BIG_FIVE_COLORS[trait]
    means = trait_agg[f"{trait}_mean"].values
    if trait in lit:
        ax.plot(scales, means, "o-", color=color, linewidth=2.2, markersize=6,
                label=trait, zorder=4)
        _draw_col_error_bars(ax, trait_agg, trait, scales, means, color)
    else:
        ax.plot(scales, means, "o-", color=color, linewidth=1.2, markersize=3,
                alpha=FAINT_ALPHA, label=trait, zorder=3)

# ── Choice mass on the same y-axis ────────────────────────────────────────────
if has_choice_mass:
    cm_agg = (
        df.groupby("scale")["_choice_mass"]
        .agg(["mean", "min", "max"])
        .reset_index()
        .sort_values("scale")
    )
    cm_scales = cm_agg["scale"].values
    cm_means  = cm_agg["mean"].values

    ax.fill_between(
        cm_scales,
        cm_agg["min"].values,
        cm_agg["max"].values,
        color="#888888", alpha=0.08, zorder=1,
    )
    ax.plot(
        cm_scales, cm_means, "s--",
        color="#888888", linewidth=1.4, markersize=3,
        alpha=0.55, label="Choice mass", zorder=2,
    )

# ── Decorations ───────────────────────────────────────────────────────────────
ax.axvline(0, color="gray", linestyle="--", linewidth=1.0, alpha=0.5, zorder=1)
ax.set_ylabel("Score (0\u20131)", fontsize=11)
ax.set_ylim(0, 1)
ax.set_xlabel(X_LABEL, fontsize=11)
if X_LIM is not None:
    ax.set_xlim(*X_LIM)
_set_scale_xticks(ax, scales)
ax.grid(True, alpha=0.25)

highlighted_name = next(iter(lit))
ax.set_title(
    f"Activation capping: trait scores vs. {X_LABEL.lower()}  "
    f"[highlight: {highlighted_name}]",
    fontsize=12, fontweight="bold",
)

if interval is not None:
    ax.errorbar([], [], yerr=1, fmt="none", color="gray", capsize=3,
                capthick=1.0, elinewidth=1.0, alpha=0.7, label=interval.label)

ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.13),
    fontsize=9, ncol=7, framealpha=0.85,
)
plt.tight_layout()
plt.show()